# Notebook 01 - Data Loading & Cleaning

## Objective

This notebook loads the Amazon Electronics metadata dataset, explores its structure,
filters and cleans the relevant products, and produces a clean 800-product sample
for the downstream LLM generation pipeline.

**Dataset:** Amazon Electronics metadata (JSONL gzip, ~1GB)
**Source:** Amazon Product Reviews dataset (He & McAuley, 2016)

### Steps
1. Load the raw JSONL file and extract relevant fields
2. Explore dataset structure and data quality
3. Normalize and clean text fields (HTML entities, tags, whitespace)
4. Filter out low-quality entries
5. Deduplicate on product title
6. Sample 800 products
7. Export clean dataset to CSV

In [1]:
import json
import gzip
import pandas as pd
import re
from tqdm import tqdm
import html

## 1 - Load Data

The raw dataset is a gzip-compressed JSONL file where each line is a JSON object
representing one product. We extract 5 fields:

- `title` : product name
- `description` : product description (can be a string or a list)
- `brand` : brand name
- `category` : list of categories, we keep the last (most specific) one
- `price` : product price (may be missing)

**Filtering at load time:** products missing either `title` or `description`
are discarded immediately to avoid loading unnecessary data into memory.

In [2]:
file_path = "../data/meta_Electronics.json.gz"
products  = []

with gzip.open(file_path, "rt") as f:
    for line in tqdm(f):
        item = json.loads(line)

        title       = item.get("title")
        description = item.get("description")
        categories  = item.get("category", [])
        brand       = item.get("brand", "")
        price       = item.get("price", None)

        # Keep only products with both title and description
        if title and description:
            products.append({
                "title"      : title,
                "description": description,
                "brand"      : brand,
                "category"   : categories[-1] if categories else "",
                "price"      : price
            })

df = pd.DataFrame(products)
print(f"Products loaded : {len(df):,}")
print(f"Columns         : {df.columns.tolist()}")

786445it [00:27, 28593.15it/s]


Products loaded : 672,392
Columns         : ['title', 'description', 'brand', 'category', 'price']


## 2 - Exploration

In [3]:
df.info()
print("\n5 Random Samples:")
df.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 672392 entries, 0 to 672391
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   title        672392 non-null  object
 1   description  672392 non-null  object
 2   brand        672392 non-null  object
 3   category     672392 non-null  object
 4   price        672392 non-null  object
dtypes: object(5)
memory usage: 25.6+ MB

5 Random Samples:


,title,description,brand,category,price
403575,Protective Lightweight Resistant Sleeve For Sa...,[<b>Neoprene Sleeve</b><br><br>Providing form-...,Vangoddy,Sleeves,$19.98
652984,Electric Magnetic Alleviate Fatigue Eye Care T...,[Specification:<br/><br/>This Electric Eye car...,OO,Headphones,
177054,Solar Portable Charger (Black),[- Solair solar power charger - Charges most c...,Solair,Tablet Accessories,
423412,Olight 18650 battery Holder for SR50 SR51,[original olight 18650 battery holder],OLIGHT,Camera,
592106,Q-See 16 Channel 1080p HD Security System with...,"[, Keep Your Home Safe with DVR Surveillance T...",Q see,Home Security Systems,


In [4]:
print("Top 10 categories:")
df["category"].value_counts().head(10)

Top 10 categories:


category
Cases                  40300
Batteries              14703
USB Cables             12082
Memory                 10906
AC Adapters            10703
Traditional Laptops    10413
Headphones             10225
Chargers & Adapters    10089
Camera Cases            9712
Earbud Headphones       9614
Name: count, dtype: int64

In [5]:
print("Description length distribution:")
df["description"].str.len().describe()

Description length distribution:


count    672392.000000
mean          1.838909
std           3.089996
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         180.000000
Name: description, dtype: float64

### Observation — Short Descriptions

A significant number of descriptions have length = 1, which is unexpected.
Inspecting these rows reveals that descriptions are stored as **lists** in the
raw JSON, not plain strings. A list like `["Great product"]` has length 1 as
a list, not as a string.

These need to be flattened before any length-based filtering.

In [6]:
print(f"Descriptions with length == 1 : {(df['description'].str.len() == 1).sum()}")
df[df["description"].str.len() == 1].head(5)

Descriptions with length == 1 : 555083


,title,description,brand,category,price
0,Genuine Geovision 1 Channel 3rd Party NVR IP S...,[The following camera brands and models have b...,GeoVision,Surveillance DVR Kits,$65.00
1,"Books ""Handbook of Astronomical Image Processi...",[This second edition of the Handbook of Astron...,33 Books Co.,Camera &amp; Photo,
5,Girl with a One-track Mind: Confessions of the...,[GIRL WITH A ONE-TRACK MIND: CONFESSIONS OF TH...,ABBY LEE,eBook Readers,$4.76
6,abcGoodefg&reg; 4GB USB 2.0 Mp3 Music Player w...,[Support system: Windows XP/Vsita/7 * SNR: 85d...,Crazy Cart,MP3 & MP4 Players,
11,SAMSUNG Evo Plus 64 GB MicroSDXC Class 10 80 M...,"[Brand SAMSUNG Speed 80 MB/s Read Speed, 20 Wr...",Samsung,Micro SD Cards,


## 3 - Cleaning

### 3.1 - Normalize Descriptions

Descriptions are stored either as plain strings or as lists of strings.
We flatten lists into a single space-joined string before applying text cleaning.

In [7]:
def normalize_description(desc):
    """Flatten list descriptions into a single string."""
    if isinstance(desc, list):
        return " ".join([str(x) for x in desc])
    if isinstance(desc, str):
        return desc
    return ""

df["description"] = df["description"].apply(normalize_description)

print("Description length after normalization:")
print(df["description"].str.len().describe())

Description length after normalization:
count    672392.000000
mean        780.093187
std        1290.394896
min           0.000000
25%         202.000000
50%         506.000000
75%        1052.000000
max      184147.000000
Name: description, dtype: float64


### 3.2 - Clean Text Fields

Applied to `title`, `description`, `brand`, `category`:
- **HTML entity decoding** : converts `&amp;` → `&`, `&#8220;` → `"`, etc.
- **HTML tag removal** : strips any remaining `<tag>` patterns
- **Whitespace normalization** : collapses multiple spaces/newlines into one
- **Strip** : removes leading and trailing whitespace

In [8]:
def clean_text(text):
    """Clean a text field by removing HTML and normalizing whitespace."""
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)          # Decode HTML entities
    text = re.sub(r"<.*?>", "", text)   # Remove HTML tags
    text = re.sub(r"\s+", " ", text)    # Normalize whitespace
    return text.strip()

df["description"] = df["description"].apply(clean_text)
df["title"]       = df["title"].apply(clean_text)
df["brand"]       = df["brand"].apply(clean_text)
df["category"]    = df["category"].apply(clean_text)

print("Text fields cleaned.")

Text fields cleaned.


### 3.3 - Filter & Deduplicate

In [9]:
initial_count = len(df)

# Remove descriptions shorter than 30 characters (not meaningful)
df = df[df["description"].str.len() > 30]
print(f"After length filter     : {len(df):,} products (removed {initial_count - len(df):,})")

# Deduplicate on title
before_dedup = len(df)
df = df.drop_duplicates(subset="title", keep="first")
print(f"After deduplication     : {len(df):,} products (removed {before_dedup - len(df):,})")

print(f"\nTotal removed           : {initial_count - len(df):,}")
print(f"Final dataset size      : {len(df):,}")

After length filter     : 653,165 products (removed 19,227)
After deduplication     : 615,623 products (removed 37,542)

Total removed           : 56,769
Final dataset size      : 615,623


In [10]:
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nUnique titles : {df['title'].nunique():,}")
df.head()

Missing values per column:
title          0
description    0
brand          0
category       0
price          0
dtype: int64

Unique titles : 615,623


,title,description,brand,category,price
0,Genuine Geovision 1 Channel 3rd Party NVR IP S...,The following camera brands and models have be...,GeoVision,Surveillance DVR Kits,$65.00
1,"Books ""Handbook of Astronomical Image Processi...",This second edition of the Handbook of Astrono...,33 Books Co.,Camera & Photo,
2,One Hot Summer,A zesty tale. (Publishers Weekly)Garcia Aguile...,Visit Amazon's Carolina Garcia Aguilera Page,eBook Readers,$11.49
3,sex.lies.murder.fame.: A Novel,“sex.lies.murder.fame. is brillllli—f@#*ing—an...,Visit Amazon's Lolita Files Page,eBook Readers,$13.95
5,Girl with a One-track Mind: Confessions of the...,GIRL WITH A ONE-TRACK MIND: CONFESSIONS OF THE...,ABBY LEE,eBook Readers,$4.76


## 4 - Sampling

We sample 800 products from the cleaned dataset using a fixed random seed
for reproducibility. `random_state=42` ensures the same sample is drawn
on every run.

800 products is sufficient for:
- Prompt engineering validation (Notebook 02)
- Evaluation with BLEU/ROUGE/cosine (Notebook 03)
- RAG-enhanced generation (Notebook 04)
- LoRA fine-tuning on a free-tier Colab GPU (Notebook 05)

In [11]:
df_sample = df.sample(800, random_state=42)
df_sample = df_sample.reset_index(drop=True)

print(f"Sample shape          : {df_sample.shape}")
print(f"Unique titles         : {df_sample['title'].nunique()}")
print(f"Missing prices        : {df_sample['price'].isna().sum()} ({df_sample['price'].isna().mean()*100:.1f}%)")
print(f"Missing brands        : {(df_sample['brand'] == '').sum()}")
print(f"Top 5 categories:")
print(df_sample["category"].value_counts().head())

Sample shape          : (800, 5)
Unique titles         : 800
Missing prices        : 0 (0.0%)
Missing brands        : 2
Top 5 categories:
category
Cases                    53
Connectors & Adapters    19
Screen Protectors        15
Sleeves                  14
Headphones               14
Name: count, dtype: int64


In [12]:
df_sample.to_csv("../data/clean_products_800.csv", index=False)
print("Dataset exported to ../data/clean_products_800.csv")

Dataset exported to ../data/clean_products_800.csv


## 5 - Summary

| Step | Detail |
|---|---|
| Raw dataset | Amazon Electronics metadata (~1GB JSONL gzip) |
| Load filter | title AND description must be present |
| Normalization | List descriptions flattened to string |
| HTML cleaning | Entities decoded, tags removed, whitespace normalized |
| Length filter | Descriptions > 30 characters only |
| Deduplication | On `title` field, keep first occurrence |
| Sample size | 800 products, `random_state=42` |
| Output | `clean_products_800.csv` |

### Limitations
- **Brand missing for some products** — same issue, kept as empty string.
- **Description quality varies** — some original Amazon descriptions are
  logistic text, compatibility lists, or generic boilerplate rather than
  actual product descriptions. This impacts evaluation metrics in Notebook 03.

### Next Step

The clean dataset is ready for **Notebook 02 — Prompt Engineering**, where
Mistral-7B-Instruct will generate short, marketing and technical descriptions
for each of the 800 products.